# Scorecard Development Demo

This notebook provides a complete, step-by-step walkthrough of building a credit scorecard using the `newt` package. We will use a large-scale real-world dataset to demonstrate how `newt` handles feature selection, optimal binning, and statistical modeling.

## Workflow Phases
1. **Environment Setup**: Importing necessary libraries.
2. **Data Loading & Sampling**: Handling large-scale parquet data.
3. **Feature Selection**: Filtering variables based on statistical criteria.
4. **Optimal Binning**: Creating monotonic bins and calculating Weight of Evidence (WOE).
5. **Modeling**: Fitting a Logistic Regression model.
6. **Scorecard Conversion**: Scaling coefficients into a point-based system.
7. **Evaluation**: Assessing the model's predictive power and stability.

In [1]:
import newt
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings

warnings.filterwarnings('ignore')
pd.set_option('display.max_columns', 100)

print(f"Newt version: {newt.__version__}")

Newt version: 0.2.0


/Users/cabbage/Project/newt/.venv/lib/python3.8/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## 1. Data Loading & Sampling

The dataset `all_data.pq` contains approximately 4 million records. For interactive development, we'll work with a sample of 500,000 records. In a production environment, you would use the full dataset.

In [2]:
DATA_PATH = 'data/test_data/all_data.pq'
df_full = pd.read_parquet(DATA_PATH)

# Sample 500k records for demonstration
df = df_full.sample(n=500000, random_state=42).copy()
print(f"Sampled dataset shape: {df.shape}")

Sampled dataset shape: (500000, 299)


### Target and Split Distribution

The dataset is pre-tagged into training, testing, and out-of-time (OOT) segments.

In [3]:
df['tag'] = df['tag'].fillna('oos')

In [4]:
print("Segment Counts:")
print(df['tag'].value_counts(dropna=False))

print("\nDefault Rate (Target=1):")
print(df['target'].value_counts(normalize=True))

Segment Counts:
tag
train    253769
oot      103261
oos       79479
test      63491
Name: count, dtype: int64

Default Rate (Target=1):
target
0.0    0.925546
1.0    0.074454
Name: proportion, dtype: float64


In [5]:
# Dataset separation
train_df = df[df['tag'] == 'train'].copy()
test_df = df[df['tag'] == 'test'].copy()
oot_df = df[df['tag'] == 'oot'].copy()

print(f"Train: {len(train_df)}, Test: {len(test_df)}, OOT: {len(oot_df)}")

Train: 253769, Test: 63491, OOT: 103261


## 2. Feature Selection

We utilize `FeatureSelector` to remove variables that are unlikely to contribute to a robust model. Criteria include:
- **Missing Rate**: Variables with >95% missing values.
- **Single Value Rate**: Variables where one value dominates >95% of records.
- **IV (Information Value)**: Variables with IV < 0.02 are considered weak predictors.
- **Correlation**: Removing one of a pair of highly correlated features (>0.7) to reduce multicollinearity.

In [6]:
target = 'target'
exclude_cols = ['idx', 'tag', 'listinginfo', 'update_time_cnt', 'update_all_cnt', 'xgb_score_p', 'xgb_score']
candidate_features = [c for c in train_df.columns if c not in [target] + exclude_cols]

# 1. 初始化并运行分析
fs = newt.FeatureSelector()
fs.fit(train_df[candidate_features], train_df[target])

# 2. 设定阈值进行筛选
fs.select(
    missing_threshold=0.95, 
    iv_threshold=0.02, 
    corr_threshold=0.7
)

# Review findings
selection_report = fs.report()
selected_features = selection_report[selection_report['status'] == 'selected']['feature'].tolist()

print(f"Retained {len(selected_features)} features after selection.")
selection_report.head(10)

Retained 52 features after selection.


,feature,dtype,count,correlation,iv,ks,kurtosis,lift_10,max,mean,median,min,missing_rate,skewness,std,status,reason
0,update_timespan,numeric,253769,-0.012017,0.007242,0.023383,129.573237,1.017721,732.0,12.853837,7.0,0.0,0.000047,9.848979,34.090199,removed,iv=0.0072
1,_educationid,numeric,253769,0.007176,0.001741,0.004926,10.318645,1.018229,6.0,0.937172,1.0,0.0,0.000047,-1.329960,0.302859,removed,iv=0.0017
2,_hasbuycar,numeric,253769,0.004647,0.000550,0.005035,9.536774,0.979105,4.0,0.937673,1.0,0.0,0.000047,-2.080596,0.277922,removed,iv=0.0005
3,_idnumber,numeric,253769,0.011110,0.001705,0.017185,-0.107640,1.105114,2.0,0.217263,0.0,0.0,0.000047,1.373256,0.412499,removed,iv=0.0017
4,_lastupdatedate,numeric,253769,0.037326,0.011454,0.053822,12.565183,1.306321,13.0,0.453193,0.0,0.0,0.000047,2.649077,0.781419,removed,iv=0.0115
5,_marriagestatusid,numeric,253769,0.007850,0.002256,0.006516,8.426819,1.001462,3.0,0.930607,1.0,0.0,0.000047,-2.212922,0.282902,removed,iv=0.0023
6,_mobilephone,numeric,253769,0.025516,0.009660,0.021714,5.551854,1.196063,5.0,0.952151,1.0,0.0,0.000047,0.456358,0.453096,removed,iv=0.0097
7,_qq,numeric,253769,0.007489,0.001030,0.007898,6.183201,0.989267,4.0,0.912227,1.0,0.0,0.000047,-2.236783,0.301886,removed,iv=0.0010
8,_realname,numeric,253769,0.019215,0.006399,0.025480,3.635528,1.163545,5.0,0.301946,0.0,0.0,0.000047,1.736375,0.526297,removed,iv=0.0064
9,_residencephone,numeric,253769,0.013036,0.005638,0.009845,12.128515,1.067006,5.0,0.949290,1.0,0.0,0.000047,-1.189717,0.288218,removed,iv=0.0056


In [7]:
selection_report

,feature,dtype,count,correlation,iv,ks,kurtosis,lift_10,max,mean,median,min,missing_rate,skewness,std,status,reason
0,update_timespan,numeric,253769,-0.012017,0.007242,0.023383,129.573237,1.017721,732.000000,12.853837,7.000000,0.000000,0.000047,9.848979,34.090199,removed,iv=0.0072
1,_educationid,numeric,253769,0.007176,0.001741,0.004926,10.318645,1.018229,6.000000,0.937172,1.000000,0.000000,0.000047,-1.329960,0.302859,removed,iv=0.0017
2,_hasbuycar,numeric,253769,0.004647,0.000550,0.005035,9.536774,0.979105,4.000000,0.937673,1.000000,0.000000,0.000047,-2.080596,0.277922,removed,iv=0.0005
3,_idnumber,numeric,253769,0.011110,0.001705,0.017185,-0.107640,1.105114,2.000000,0.217263,0.000000,0.000000,0.000047,1.373256,0.412499,removed,iv=0.0017
4,_lastupdatedate,numeric,253769,0.037326,0.011454,0.053822,12.565183,1.306321,13.000000,0.453193,0.000000,0.000000,0.000047,2.649077,0.781419,removed,iv=0.0115
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
286,flag_oot,numeric,253769,NaN,0.000000,0.000000,NaN,0.992832,0.000000,0.000000,0.000000,0.000000,0.000000,NaN,0.000000,removed,iv=0.0000
287,flag_oos,numeric,253769,NaN,0.000000,0.000000,NaN,0.992832,0.000000,0.000000,0.000000,0.000000,0.000000,NaN,0.000000,removed,iv=0.0000
288,flag_test,numeric,253769,NaN,0.000000,0.000000,NaN,0.992832,0.000000,0.000000,0.000000,0.000000,0.000000,NaN,0.000000,removed,iv=0.0000
289,score_p,numeric,253769,0.247661,0.779365,0.343524,3.265068,2.960204,0.433002,0.077856,0.065497,0.008463,0.000000,1.570252,0.047479,selected,


## 3. Optimal Binning

The `Binner` class calculates optimal cut points. We use **ChiMerge** and enforce **monotonicity**, which is a requirement for most credit scoring models to ensure the risk trend follows a logical direction.

In [8]:
binner = newt.Binner()
binner.fit(
    train_df[selected_features], 
    train_df[target], 
    method='opt', 
    n_bins=10, 
    monotonic=True
)

Binning features: 100%|███████████████████████████████████████████████████████| 36/36 [00:03<00:00, 11.71it/s]


### Visualizing Feature Bins
Let's visualize the binning result for a specific feature, e.g., `_age`.

In [9]:
print(binner.rules_.keys())

dict_keys(['log_cnt', 'userinfo_1', 'userinfo_3', 'userinfo_14', 'userinfo_16', 'userinfo_17', 'webloginfo_2', 'webloginfo_5', 'webloginfo_6', 'webloginfo_7', 'webloginfo_15', 'education_info1', 'education_info5', 'thirdparty_info_period1_3', 'thirdparty_info_period1_9', 'thirdparty_info_period1_10', 'thirdparty_info_period1_15', 'thirdparty_info_period2_2', 'thirdparty_info_period2_6', 'thirdparty_info_period2_10', 'thirdparty_info_period2_13', 'thirdparty_info_period2_15', 'thirdparty_info_period3_10', 'thirdparty_info_period4_8', 'thirdparty_info_period4_10', 'thirdparty_info_period5_2', 'thirdparty_info_period5_4', 'thirdparty_info_period5_10', 'thirdparty_info_period5_17', 'thirdparty_info_period6_1', 'thirdparty_info_period6_5', 'thirdparty_info_period6_6', 'thirdparty_info_period6_8', 'thirdparty_info_period6_10', 'thirdparty_info_period6_14', 'score_p'])


In [10]:
selected_features = ['log_cnt', 'userinfo_1', 'userinfo_3', 'userinfo_14', 'userinfo_16', 'userinfo_17', 'webloginfo_2', 'webloginfo_5', 'webloginfo_6', 'webloginfo_7', 'webloginfo_15', 'education_info1', 'thirdparty_info_period1_3', 'thirdparty_info_period1_9', 'thirdparty_info_period1_10', 'thirdparty_info_period1_15', 'thirdparty_info_period2_2', 'thirdparty_info_period2_6', 'thirdparty_info_period2_10', 'thirdparty_info_period2_13', 'thirdparty_info_period2_15', 'thirdparty_info_period3_10', 'thirdparty_info_period4_8', 'thirdparty_info_period4_10', 'thirdparty_info_period5_2', 'thirdparty_info_period5_4', 'thirdparty_info_period5_10', 'thirdparty_info_period5_17', 'thirdparty_info_period6_1', 'thirdparty_info_period6_5', 'thirdparty_info_period6_6', 'thirdparty_info_period6_8', 'thirdparty_info_period6_10', 'thirdparty_info_period6_14']

In [11]:
# if '_age' in binner.features():
#     binner['_age'].plot(title="Age Binning Results")

In [12]:
# binner['userinfo_2'].plot()

## 4. WOE Transformation & Modeling

We transform the raw data into Weight of Evidence (WOE) values. WOE makes the relationship between a feature and the target linear, which is ideal for Logistic Regression.

In [13]:
X_train_woe = binner.woe_transform(train_df[selected_features])
X_test_woe = binner.woe_transform(test_df[selected_features])
X_oot_woe = binner.woe_transform(oot_df[selected_features])

y_train = train_df[target]
y_test = test_df[target]
y_oot = oot_df[target]

### Logistic Regression Fit
The `LogisticModel` wrapper provides full statistical summaries including P-values for variable significance testing.

In [14]:
lr_model = newt.LogisticModel()
lr_model.fit(X_train_woe[selected_features], y_train)

print(lr_model.summary())

/Users/cabbage/Project/newt/.venv/lib/python3.8/site-packages/statsmodels/base/model.py:607: ConvergenceWarning: Maximum Likelihood optimization failed to converge. Check mle_retvals
  warnings.warn("Maximum Likelihood optimization failed to "


                           Logit Regression Results                           
Dep. Variable:                 target   No. Observations:               253769
Model:                          Logit   Df Residuals:                   253734
Method:                           MLE   Df Model:                           34
Date:                Sun, 12 Apr 2026   Pseudo R-squ.:                 0.09056
Time:                        08:56:13   Log-Likelihood:                -62948.
converged:                      False   LL-Null:                       -69217.
Covariance Type:            nonrobust   LLR p-value:                     0.000
                                 coef    std err          z      P>|z|      [0.025      0.975]
----------------------------------------------------------------------------------------------
const                         -2.4763      0.008   -306.702      0.000      -2.492      -2.460
log_cnt                        0.2971      0.100      2.968      0.003       0.101 

## 5. Scorecard Generation

We convert the regression model into a point-based scorecard using standard scaling parameters:
- **Base Score (600)**: The score representing the base odds.
- **PDO (20)**: Points to Double the Odds (e.g., if a score of 600 has 1:50 odds, 620 has 2:50 odds).

In [15]:
scorecard = newt.Scorecard(base_score=600, pdo=20)
scorecard.from_model(lr_model, binner)

print(scorecard.summary())

Scorecard Summary
Base Score: 600
PDO: 20
Base Odds: 1.0000
Factor: 28.8539
Offset: 600.0000
Intercept Points: 671.45
Number of Features: 34
--------------------------------------------------
Features:
  log_cnt: 7 bins, points range [-1.0, 3.2]
  userinfo_1: 3 bins, points range [-1.6, 19.7]
  userinfo_3: 4 bins, points range [-1.6, 27.1]
  userinfo_14: 4 bins, points range [-8.8, 13.4]
  userinfo_16: 3 bins, points range [-4.6, 13.7]
  userinfo_17: 2 bins, points range [-1.3, 11.9]
  webloginfo_2: 3 bins, points range [-6.5, 1.4]
  webloginfo_5: 5 bins, points range [-0.3, 1.1]
  webloginfo_6: 7 bins, points range [-16.0, 4.5]
  webloginfo_7: 4 bins, points range [-0.2, 0.9]
  webloginfo_15: 5 bins, points range [-8.5, 1.9]
  education_info1: 2 bins, points range [-0.9, 22.6]
  thirdparty_info_period1_3: 9 bins, points range [-4.7, 2.7]
  thirdparty_info_period1_9: 5 bins, points range [-1.1, 5.0]
  thirdparty_info_period1_10: 5 bins, points range [-1.9, 10.0]
  thirdparty_info_perio

## 6. Performance Evaluation

Applying the scorecard to our datasets and calculating key performance indicators (KPIs).

In [16]:
train_score = scorecard.score(train_df)
test_score = scorecard.score(test_df)
oot_score = scorecard.score(oot_df)

def evaluate(name, y_true, score):
    # Higher score should mean lower risk for Newt core scorecard scaling
    # AUC/KS usually expect risk probability, so we negate the score
    auc = newt.calculate_auc(y_true, -score)
    ks = newt.calculate_ks(y_true, -score)
    return {'Dataset': name, 'AUC': round(auc, 4), 'KS': round(ks, 4)}

summary_metrics = pd.DataFrame([
    evaluate('Train', y_train, train_score),
    evaluate('Test', y_test, test_score),
    evaluate('OOT', y_oot, oot_score)
])

summary_metrics

,Dataset,AUC,KS
0,Train,0.7268,0.3313
1,Test,0.7220,0.3245
2,OOT,0.7353,0.3648


### Stability Analysis (PSI)

PSI monitors whether our score distribution is shifting over time.

In [17]:
psi_test = newt.calculate_psi(train_score, test_score)
psi_oot = newt.calculate_psi(train_score, oot_score)

print(f"PSI (Train vs Test): {psi_test:.4f}")
print(f"PSI (Train vs OOT): {psi_oot:.4f}")

PSI (Train vs Test): 0.0001
PSI (Train vs OOT): 0.0102


## 7. Results Export

Finally, we can export the scorecard as a flat table for production deployment or further review.

In [18]:
score_table = scorecard.export()
score_table

,feature,bin,woe,points
0,Intercept,-,0.000000,671.451423
1,log_cnt,"(-inf, 13.5]",0.112715,-0.966246
2,log_cnt,"(13.5, 15.5]",0.095356,-0.817439
3,log_cnt,"(15.5, 22.5]",-0.013506,0.115783
4,log_cnt,"(22.5, 30.5]",-0.101425,0.869461
...,...,...,...,...
164,thirdparty_info_period6_10,"(2.5, inf]",0.854740,-2.939505
165,thirdparty_info_period6_14,"(-inf, 976.5]",-0.299196,0.682919
166,thirdparty_info_period6_14,"(11736.5, inf]",0.230830,-0.526873
167,thirdparty_info_period6_14,"(3547.5, 11736.5]",0.189899,-0.433447


In [19]:
# df_woe = binner.woe_transform(df[selected_features])

In [20]:
score = scorecard.score(df)

In [21]:
df['score_lr'] = score

In [22]:
# df

In [23]:
from newt import Report

# 初始化报告配置
report = Report(
    data=df,                # 包含分数、特征和标签的 DataFrame
    model=scorecard,          # 训练好的模型对象
    tag="tag",               # 样本集区分列（如 train/test/oot）
    score_col="score_lr",       # 评分卡得出的分数列名
    date_col="listinginfo",  # 日期列（用于做月度趋势分析）
    label_list=["target"],   # 目标变量列表
    report_out_path="./model_report.xlsx"  # 报告保存路径
)

# 生成报告
output_path = report.generate()

2026-04-12 08:56:20,052 | DEBUG | newt.reporting.report | Report generation started | rows=500000 cols=300 primary_score=score_lr labels=['target'] output=./model_report.xlsx engine=rust workers=8 parallel_sheets=True memory_mode=compact metrics_mode=exact peak_rss_mb=3093.1
2026-04-12 08:56:23,906 | DEBUG | newt.reporting.report | Step prepare_data finished | elapsed=3.852s | rows=500000 peak_rss_mb=3093.1
2026-04-12 08:56:24,695 | DEBUG | newt.reporting.report | Step prepare_report_scores finished | elapsed=0.786s | report_scores=['score_lr'] peak_rss_mb=3093.1
2026-04-12 08:56:24,696 | DEBUG | newt.reporting.report | Step resolve_sheet_keys finished | elapsed=0.000s | selected_sheet_keys=['overview', 'model_design', 'variable_analysis', 'scorecard_details', 'model_performance', 'dimensional_comparison', 'model_comparison', 'portrait']
2026-04-12 08:56:24,696 | DEBUG | newt.reporting.report | Step model_adapter_init finished | elapsed=0.000s | model_family=scorecard
2026-04-12 08:56:

In [24]:
import pickle

In [25]:
with open('./data/test_data/lr_model.pkl', 'wb') as f:
    pickle.dump(scorecard, f)

In [26]:
df.to_parquet('./data/test_data/lr_data.pq')